# 추천 알고리즘 — 모델 선정 스파이크

**스파이크/평가용 코드입니다. 프로덕션 코드가 아닙니다.**

상품명/설명 임베딩 기반 후보 생성과, LLM zero-shot 재정렬 후보 모델을 실제로 로드해서 비교합니다.

- 후보 생성(콘텐츠 기반): [`kakaocorp/kanana-nano-2.1b-embedding`](https://huggingface.co/kakaocorp/kanana-nano-2.1b-embedding)
- 범용 비교 baseline: [`BAAI/bge-m3`](https://huggingface.co/BAAI/bge-m3)
- 개인화 재정렬: `kanana-1.5-instruct` 재사용 (댓글 관리 노트북과 동일한 로더)

모델을 바꾸려면 `model_registry.py`의 `EMBEDDING_MODELS` / `ACTIVE_GENERATIVE` 값만 바꾸고 해당 셀들을 다시 실행하면 됩니다.

In [1]:
%pip install -q -r ../requirements-test.txt

Note: you may need to restart the kernel to use updated packages.


## 1. kanana-nano-2.1b-embedding (한국어 특화)

In [2]:
from model_registry import EMBEDDING_MODELS
from loaders import load_kanana_embedding

kanana_emb = load_kanana_embedding(EMBEDDING_MODELS["kanana-nano-embedding"])
print(f"loaded: {kanana_emb.name} on device={kanana_emb.device}")

QUERY_INSTRUCTION = "Given a question, retrieve passages that answer the question"
DOC_INSTRUCTION = ""

query = "빈티지 데님 자켓"
products = [
    "90년대 리바이스 오버사이즈 데님 자켓",
    "구제 코듀로이 반팔 셔츠",
    "빈티지 워크웨어 스타일 청자켓",
    "여름용 린넨 원피스",
    "스트릿 그래픽 후드티",
]

kanana_query_vec = kanana_emb.model.encode(
    [query], instruction=QUERY_INSTRUCTION, max_length=512
)
kanana_product_vecs = kanana_emb.model.encode(
    products, instruction=DOC_INSTRUCTION, max_length=512
)

/Users/who._.hs/opt/S15P11A108/AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded: kakaocorp/kanana-nano-2.1b-embedding on device=mps


In [3]:
import torch
import torch.nn.functional as F

q = torch.as_tensor(kanana_query_vec)
p = torch.as_tensor(kanana_product_vecs)
kanana_sims = F.cosine_similarity(q, p)

kanana_ranked = sorted(
    zip(products, kanana_sims.tolist()), key=lambda x: x[1], reverse=True
)
for name, score in kanana_ranked:
    print(f"{score:.3f}  {name}")

0.683  빈티지 워크웨어 스타일 청자켓
0.680  90년대 리바이스 오버사이즈 데님 자켓
0.544  여름용 린넨 원피스
0.537  구제 코듀로이 반팔 셔츠
0.509  스트릿 그래픽 후드티


## 2. bge-m3 비교 baseline

In [4]:
from loaders import load_bge_m3

bge = load_bge_m3(EMBEDDING_MODELS["bge-m3"])

bge_query_vec = bge.encode([query])["dense_vecs"]
bge_product_vecs = bge.encode(products)["dense_vecs"]

bq = torch.as_tensor(bge_query_vec)
bp = torch.as_tensor(bge_product_vecs)
bge_sims = F.cosine_similarity(bq, bp)

bge_ranked = sorted(zip(products, bge_sims.tolist()), key=lambda x: x[1], reverse=True)

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 370085.65it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


## 3. 두 임베딩 모델 랭킹 비교

In [5]:
print("kanana-nano-embedding 랭킹:")
for name, score in kanana_ranked:
    print(f"  {score:.3f}  {name}")

print("\nbge-m3 랭킹:")
for name, score in bge_ranked:
    print(f"  {score:.3f}  {name}")

kanana-nano-embedding 랭킹:
  0.683  빈티지 워크웨어 스타일 청자켓
  0.680  90년대 리바이스 오버사이즈 데님 자켓
  0.544  여름용 린넨 원피스
  0.537  구제 코듀로이 반팔 셔츠
  0.509  스트릿 그래픽 후드티

bge-m3 랭킹:
  0.714  90년대 리바이스 오버사이즈 데님 자켓
  0.696  빈티지 워크웨어 스타일 청자켓
  0.544  여름용 린넨 원피스
  0.519  스트릿 그래픽 후드티
  0.466  구제 코듀로이 반팔 셔츠


## 4. kanana-1.5-instruct 재정렬 — 댓글 관리 노트북의 로더 재사용 (중복 코드 없음)

In [6]:
from model_registry import GENERATIVE_MODELS, ACTIVE_GENERATIVE
from loaders import load_causal_lm, generate_chat

gen_bundle = load_causal_lm(GENERATIVE_MODELS[ACTIVE_GENERATIVE])

candidates_text = "\n".join(f"{i + 1}. {name}" for i, name in enumerate(products))
rerank_prompt = (
    f"사용자가 '{query}'를 찾고 있다. 아래 상품 후보를 관련성이 높은 순서대로 "
    f"번호만 나열해서 재정렬하라.\n\n{candidates_text}"
)
print(generate_chat(gen_bundle, rerank_prompt))

관련성이 높은 순서대로 번호를 재정렬하면:

3. 빈티지 워크웨어 스타일 청자켓  
1. 90년대 리바이스 오버사이즈 데님 자켓  
5. 스트릿 그래픽 후드티


인기 기반 랭킹(popularity-based ranking)은 조회수/판매량 같은 단순 카운트 로직으로, 콜드스타트 fallback 용도입니다. 로드할 모델이 없어 이 노트북의 대상이 아닙니다.